In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import remfile

from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO


# 当前Notebook应该位于：
# D:\Projects\ibl-population-dynamics\notebooks
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

RESULTS_DIR = PROJECT_ROOT / "results"
CACHE_DIR = PROJECT_ROOT / "cache" / "remfile"
DATA_DIR = PROJECT_ROOT / "data"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook directory:", NOTEBOOK_DIR)
print("Project root:", PROJECT_ROOT)
print("Cache directory:", CACHE_DIR)

Notebook directory: D:\Ephys_analysis\ibl-population-dynamics\notebooks
Project root: D:\Ephys_analysis\ibl-population-dynamics
Cache directory: D:\Ephys_analysis\ibl-population-dynamics\cache\remfile


In [2]:
inventory_path = RESULTS_DIR / "session_inventory.csv"

session_inventory = pd.read_csv(inventory_path)

print("Inventory path:", inventory_path)
print("Inventory shape:", session_inventory.shape)

display(session_inventory.head())

Inventory path: D:\Ephys_analysis\ibl-population-dynamics\results\session_inventory.csv
Inventory shape: (459, 3)


,subject,session,asset_path
0,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...
1,CSH-ZAD-011,03cf52f6-fba6-4743-a42e-dd1ac3072343,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-03cf52f6-f...
2,CSH-ZAD-011,5b44c40f-80f4-44fb-abfb-c7f19e27a6ca,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-5b44c40f-8...
3,CSH-ZAD-011,da188f2c-553c-4e04-879b-c9ea2d1b9a93,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-da188f2c-5...
4,CSH-ZAD-019,49e0ab27-827a-4c91-bcaa-97eea27a1b8d,sub-CSH-ZAD-019/sub-CSH-ZAD-019_ses-49e0ab27-8...


In [3]:
DANDISET_ID = "000409"
VERSION = "0.260309.1324"

preferred_session = "6ed57216-498d-48a6-b48b-a243a34710ea"

matched = session_inventory.loc[
    session_inventory["session"] == preferred_session
]

if len(matched) > 0:
    test_row = matched.iloc[0]
else:
    print("Preferred session not found; using the first session.")
    test_row = session_inventory.iloc[0]

test_subject = test_row["subject"]
test_session = test_row["session"]
test_asset_path = test_row["asset_path"]

print("Subject:", test_subject)
print("Session:", test_session)
print("Asset path:", test_asset_path)

Subject: NYU-39
Session: 6ed57216-498d-48a6-b48b-a243a34710ea
Asset path: sub-NYU-39/sub-NYU-39_ses-6ed57216-498d-48a6-b48b-a243a34710ea_desc-processed_behavior+ecephys.nwb


In [4]:
with DandiAPIClient() as client:
    dandiset = client.get_dandiset(
        dandiset_id=DANDISET_ID,
        version_id=VERSION,
    )

    asset = dandiset.get_asset_by_path(
        path=test_asset_path
    )

    s3_url = asset.get_content_url(
        follow_redirects=1,
        strip_query=True,
    )

print("Remote URL obtained.")
print(s3_url[:120], "...")

Remote URL obtained.
https://dandiarchive.s3.amazonaws.com/blobs/8d5/f43/8d5f4360-984d-4117-bd2d-e86263e7e3a4 ...


In [5]:
disk_cache = remfile.DiskCache(str(CACHE_DIR))

rem_file = remfile.File(
    s3_url,
    disk_cache=disk_cache,
)

h5py_file = h5py.File(
    rem_file,
    mode="r",
)

io = NWBHDF5IO(
    file=h5py_file,
    load_namespaces=True,
)

nwbfile = io.read()

print("NWB opened successfully.")
print("Session ID:", nwbfile.session_id)
print("Number of trials:", len(nwbfile.trials))
print("Number of units:", len(nwbfile.units))

NWB opened successfully.
Session ID: 6ed57216-498d-48a6-b48b-a243a34710ea
Number of trials: 541
Number of units: 1366


In [6]:
# 建立trails的数据框
import numpy as np
import pandas as pd

trials_df = (
    nwbfile.trials
    .to_dataframe()
    .reset_index()
    .rename(columns={"id": "trial_index", "index": "trial_index"})
)

print("Trials shape:", trials_df.shape)
display(trials_df.head())

Trials shape: (541, 18)


,trial_index,start_time,stop_time,quiescence_period,gabor_stimulus_onset_time,auditory_cue_time,wheel_movement_onset_time,choice_registration_time,feedback_time,gabor_stimulus_offset_time,gabor_stimulus_contrast,gabor_stimulus_side,mouse_wheel_choice,is_mouse_rewarded,reward_volume_uL,probability_left,block_type,block_index
0,0,49.622837,52.025851,0.441705,50.242575,50.243408,50.330254,50.456541,50.456654,51.525847,100.0,left,clockwise,True,1.5,0.5,unbiased,0
1,1,52.488151,56.125782,0.410087,54.230543,54.231410,54.416254,54.558867,54.558977,55.625737,12.5,right,counter_clockwise,True,1.5,0.5,unbiased,0
2,2,56.503582,58.792414,0.446345,57.014298,57.015398,57.136254,57.224792,57.224877,58.292337,25.0,left,clockwise,True,1.5,0.5,unbiased,0
3,3,59.156617,61.480618,0.467037,59.697496,59.698396,59.722254,59.917717,59.917808,60.980601,100.0,right,counter_clockwise,True,1.5,0.5,unbiased,0
4,4,61.841715,64.309232,0.479629,62.475552,62.476385,62.355254,62.753921,62.754024,63.809118,0.0,left,clockwise,True,1.5,0.5,unbiased,0


In [7]:
categorical_trial_columns = [
    "gabor_stimulus_contrast",
    "gabor_stimulus_side",
    "mouse_wheel_choice",
    "is_mouse_rewarded",
    "probability_left",
    "block_type",
]

for column in categorical_trial_columns:
    print(f"\n{column}")
    print(trials_df[column].value_counts(dropna=False).sort_index())


gabor_stimulus_contrast
gabor_stimulus_contrast
0.00       63
6.25      108
12.50     121
25.00     122
100.00    127
Name: count, dtype: int64

gabor_stimulus_side
gabor_stimulus_side
left     308
right    233
Name: count, dtype: int64

mouse_wheel_choice
mouse_wheel_choice
clockwise            269
counter_clockwise    272
Name: count, dtype: int64

is_mouse_rewarded
is_mouse_rewarded
False     69
True     472
Name: count, dtype: int64

probability_left
probability_left
0.2    175
0.5     90
0.8    276
Name: count, dtype: int64

block_type
block_type
left_block     276
right_block    175
unbiased        90
Name: count, dtype: int64


In [8]:
#QC
trials_df["reaction_time_s"] = (
    trials_df["wheel_movement_onset_time"]
    - trials_df["gabor_stimulus_onset_time"]
)

In [9]:
required_trial_columns = [
    "gabor_stimulus_onset_time",
    "wheel_movement_onset_time",
    "choice_registration_time",
    "feedback_time",
    "mouse_wheel_choice",
    "probability_left",
    "is_mouse_rewarded",
]

trials_df["has_required_events"] = (
    trials_df[required_trial_columns]
    .notna()
    .all(axis=1)
)

In [10]:
MIN_RT_S = 0.08
MAX_RT_S = 2.00

trials_df["rt_in_range"] = trials_df["reaction_time_s"].between(
    MIN_RT_S,
    MAX_RT_S,
    inclusive="both",
)

In [11]:
trials_df["is_valid_trial"] = (
    trials_df["has_required_events"]
    & trials_df["rt_in_range"]
)

valid_trials_df = (
    trials_df.loc[trials_df["is_valid_trial"]]
    .copy()
    .reset_index(drop=True)
)

print("All trials:", len(trials_df))
print("Valid trials:", len(valid_trials_df))
print(
    "Retained fraction:",
    f"{len(valid_trials_df) / len(trials_df):.1%}",
)

All trials: 541
Valid trials: 439
Retained fraction: 81.1%


In [12]:
trial_qc_summary = pd.Series(
    {
        "all_trials": len(trials_df),
        "missing_required_events": (
            ~trials_df["has_required_events"]
        ).sum(),
        "reaction_time_out_of_range": (
            trials_df["has_required_events"]
            & ~trials_df["rt_in_range"]
        ).sum(),
        "valid_trials": trials_df["is_valid_trial"].sum(),
    },
    name="n_trials",
)

display(trial_qc_summary.to_frame())

,n_trials
all_trials,541
missing_required_events,0
reaction_time_out_of_range,102
valid_trials,439


In [13]:
#建立unit_metadata_df
light_unit_columns = [
    "unit_name",
    "cluster_uuid",
    "probe_name",
    "kilosort2_label",
    "ibl_quality_score",
    "firing_rate",
    "spike_count",
    "presence_ratio",
    "isi_violations_ratio",
    "rp_violation",
    "sliding_rp_violation",
    "noise_cutoff",
    "missed_spikes_estimate",
    "median_spike_amplitude_uV",
    "max_spike_amplitude_uV",
    "cumulative_drift_um_per_hour",
]

unit_metadata_dict = {
    "unit_index": np.arange(len(nwbfile.units))
}

for column in light_unit_columns:
    unit_metadata_dict[column] = np.asarray(
        nwbfile.units[column][:]
    )

unit_metadata_df = pd.DataFrame(unit_metadata_dict)

print("Unit metadata shape:", unit_metadata_df.shape)
display(unit_metadata_df.head())

Unit metadata shape: (1366, 17)


,unit_index,unit_name,cluster_uuid,probe_name,kilosort2_label,ibl_quality_score,firing_rate,spike_count,presence_ratio,isi_violations_ratio,rp_violation,sliding_rp_violation,noise_cutoff,missed_spikes_estimate,median_spike_amplitude_uV,max_spike_amplitude_uV,cumulative_drift_um_per_hour
0,0,probe00_0,8a3fb319-d1eb-445b-9d7f-c208067543cc,Probe00,mua,0.666667,9.986225,36327,0.983562,0.594632,0.396912,1.0,10.514859,0.046194,253.211534,370.461563,5.332562e+05
1,1,probe00_1,b1aa8503-6e2a-4e42-be94-0057db48ec2e,Probe00,mua,0.333333,31.913752,116093,0.997260,1.040543,0.605311,0.0,405.187456,0.348336,94.285195,225.289366,4.910825e+06
2,2,probe00_2,57a200c5-e79b-4346-9c64-311cc8553610,Probe00,mua,0.333333,25.103975,91321,0.997260,0.574396,0.386672,0.0,260.530651,0.500000,89.135268,216.256238,3.504317e+06
3,3,probe00_3,da179590-df96-455d-8f76-71df8d6e572f,Probe00,mua,0.666667,33.853983,123151,0.997260,0.074955,0.065618,1.0,107.066668,0.500000,80.807761,166.567349,4.660627e+06
4,4,probe00_4,7f94d191-a2e7-487f-89f7-4564bb6326d4,Probe00,mua,0.666667,24.741659,90003,0.997260,0.099606,0.086135,1.0,346.026570,0.500000,54.534394,299.106038,2.090179e+06


In [14]:
from tqdm import tqdm

def get_representative_electrode(unit_index):
    """
    Return one representative electrode row for a unit.

    Priority:
    1. max_electrode reference
    2. associated electrodes, only if their locations agree
    """
    max_ref = nwbfile.units["max_electrode"][unit_index]

    if isinstance(max_ref, pd.DataFrame) and len(max_ref) > 0:
        return max_ref.iloc[0]

    if isinstance(max_ref, pd.Series):
        return max_ref

    linked = nwbfile.units["electrodes"][unit_index]

    if isinstance(linked, pd.DataFrame) and len(linked) > 0:
        unique_locations = (
            linked["location"]
            .dropna()
            .astype(str)
            .unique()
        )

        if len(unique_locations) == 1:
            return linked.iloc[0]

    return None

In [15]:
unit_regions = []
unit_electrode_names = []
unit_channel_names = []
unit_x = []
unit_y = []
unit_z = []

for unit_index in tqdm(
    range(len(nwbfile.units)),
    desc="Mapping units to regions",
):
    electrode_row = get_representative_electrode(unit_index)

    if electrode_row is None:
        unit_regions.append(np.nan)
        unit_electrode_names.append(np.nan)
        unit_channel_names.append(np.nan)
        unit_x.append(np.nan)
        unit_y.append(np.nan)
        unit_z.append(np.nan)
        continue

    unit_regions.append(electrode_row.get("location", np.nan))
    unit_electrode_names.append(
        electrode_row.get("electrode_name", np.nan)
    )
    unit_channel_names.append(
        electrode_row.get("channel_name", np.nan)
    )
    unit_x.append(electrode_row.get("x", np.nan))
    unit_y.append(electrode_row.get("y", np.nan))
    unit_z.append(electrode_row.get("z", np.nan))

unit_metadata_df["region"] = unit_regions
unit_metadata_df["max_electrode_name"] = unit_electrode_names
unit_metadata_df["channel_name"] = unit_channel_names
unit_metadata_df["x"] = unit_x
unit_metadata_df["y"] = unit_y
unit_metadata_df["z"] = unit_z

Mapping units to regions: 100%|██████████| 1366/1366 [00:01<00:00, 825.98it/s]


In [16]:
print(
    "Units with region:",
    unit_metadata_df["region"].notna().sum(),
    "/",
    len(unit_metadata_df),
)

display(
    unit_metadata_df[
        [
            "unit_index",
            "unit_name",
            "probe_name",
            "region",
            "max_electrode_name",
            "ibl_quality_score",
        ]
    ].head(10)
)

Units with region: 1366 / 1366


,unit_index,unit_name,probe_name,region,max_electrode_name,ibl_quality_score
0,0,probe00_0,Probe00,Posterior complex of the thalamus,e0,0.666667
1,1,probe00_1,Probe00,Posterior complex of the thalamus,e0,0.333333
2,2,probe00_2,Probe00,Posterior complex of the thalamus,e1,0.333333
3,3,probe00_3,Probe00,Posterior complex of the thalamus,e5,0.666667
4,4,probe00_4,Probe00,Posterior complex of the thalamus,e6,0.666667
5,5,probe00_5,Probe00,Posterior complex of the thalamus,e6,0.333333
6,6,probe00_6,Probe00,Posterior complex of the thalamus,e6,0.666667
7,7,probe00_7,Probe00,Posterior complex of the thalamus,e6,0.333333
8,8,probe00_8,Probe00,Posterior complex of the thalamus,e6,0.333333
9,9,probe00_9,Probe00,Posterior complex of the thalamus,e6,0.333333


In [17]:
print(
    "ibl_quality_score description:\n",
    nwbfile.units["ibl_quality_score"].description,
)

print("\nValue counts:")
print(
    unit_metadata_df["ibl_quality_score"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nkilosort2_label:")
print(
    unit_metadata_df["kilosort2_label"]
    .value_counts(dropna=False)
)

ibl_quality_score description:
 Proportion of IBL quality metrics passed (0.0, 0.33, 0.67, or 1.0). 1.0 = all three passed.

Value counts:
ibl_quality_score
0.000000    267
0.333333    582
0.666667    361
1.000000    156
Name: count, dtype: int64

kilosort2_label:
kilosort2_label
mua     1077
good     289
Name: count, dtype: int64


In [18]:
region_counts_df = (
    unit_metadata_df
    .dropna(subset=["region"])
    .groupby("region", as_index=False)
    .agg(
        n_all_units=("unit_index", "size"),
        n_good_units=(
            "ibl_quality_score",
            lambda values: values.eq(1.0).sum(),
        ),
        n_probes=("probe_name", "nunique"),
        median_firing_rate_hz=("firing_rate", "median"),
    )
    .sort_values(
        ["n_good_units", "n_all_units"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(region_counts_df.head(30))

,region,n_all_units,n_good_units,n_probes,median_firing_rate_hz
0,Posterior complex of the thalamus,168,36,1,9.736755
1,Motor nucleus of trigeminal,347,28,1,4.045678
2,Parabrachial nucleus,203,25,1,3.849950
3,Lateral posterior nucleus of the thalamus,103,24,1,7.036568
4,Cuneiform nucleus,128,14,1,2.282893
5,Parvicellular reticular nucleus,56,8,1,6.997948
6,Anterior area layer 6a,28,2,1,0.582097
7,Medulla,25,2,1,4.569359
8,superior cerebelar peduncles,17,2,1,27.017278
9,Dentate gyrus polymorph layer,16,2,1,10.935036


In [19]:
population_candidate_regions = region_counts_df.query(
    "n_good_units >= 30"
)

display(population_candidate_regions)

,region,n_all_units,n_good_units,n_probes,median_firing_rate_hz
0,Posterior complex of the thalamus,168,36,1,9.736755


In [20]:
dataframes = {
    "valid_trials_df": valid_trials_df,
    "unit_metadata_df": unit_metadata_df,
    "region_counts_df": region_counts_df,
}

for name, dataframe in dataframes.items():
    memory_mb = (
        dataframe.memory_usage(deep=True).sum()
        / 1024**2
    )

    print(
        f"{name}: "
        f"{dataframe.shape}, "
        f"{memory_mb:.3f} MB"
    )

valid_trials_df: (439, 22), 0.135 MB
unit_metadata_df: (1366, 23), 0.813 MB
region_counts_df: (29, 5), 0.003 MB
